### Data Ingestion

1. Document data structure

In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="main text content",
    metadata={
        "source":"example.txt",
        "pages":1
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1}, page_content='main text content')

In [9]:
import os
os.makedirs("../Data/text_file", exist_ok=True)

In [4]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
#from pypdf import PdfReader

dir_loader = DirectoryLoader(
    "../Data/pdf",
    glob="Ma_Siu_Lun,_Ng_Kah_Loon,_Victor_Tan_Linear_Algebra_Concepts_and.pdf",
    loader_cls=PyPDFLoader,
    show_progress=False
)
document = dir_loader.load()
print(document)

d:\LangGraph\langgraph_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'pikepdf 4.3.1', 'creator': 'ocrmypdf 13.2.0 / Tesseract OCR-PDF 4.1.1', 'creationdate': '2021-08-05T14:17:12+00:00', 'author': '', 'moddate': '2022-01-08T20:19:31+00:00', 'source': '..\\Data\\pdf\\Ma_Siu_Lun,_Ng_Kah_Loon,_Victor_Tan_Linear_Algebra_Concepts_and.pdf', 'total_pages': 246, 'page': 0, 'page_label': 'i'}, page_content='LINEAR ALGEBRA \nConcepts and Techniques on Euclidean Spaces \nSecond Edition \nMa Siu Lun \nNg Kah Loon \nVictor Tan \nMc \n[\\ \nR \n[Ty'), Document(metadata={'producer': 'pikepdf 4.3.1', 'creator': 'ocrmypdf 13.2.0 / Tesseract OCR-PDF 4.1.1', 'creationdate': '2021-08-05T14:17:12+00:00', 'author': '', 'moddate': '2022-01-08T20:19:31+00:00', 'source': '..\\Data\\pdf\\Ma_Siu_Lun,_Ng_Kah_Loon,_Victor_Tan_Linear_Algebra_Concepts_and.pdf', 'total_pages': 246, 'page': 1, 'page_label': 'ii'}, page_content='Education \nLinear Algebra \nConcepts and Techniques on Euclidean Spaces \nSecond Edition \nCopyright © 2016 by McGraw-Hill Educ

### Chunking

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=500, chunk_overlap=50):
    """ Split documents into smaller chunks
    Args:
        documents: List of Document objects or raw strings.
        chunk_size: Max characters per chunk.
        chunk_overlap: Overlap between chunks.
    Returns:
        List of Document chunks
    """
    # ✅ Ensure all inputs are Document objects
    if isinstance(documents[0], str):
        documents = [Document(page_content=doc, metadata={}) for doc in documents]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # show example of chunk
    if split_docs:
        print(f"\n Example chunk")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")
    
    return split_docs

# all_pdf_documents = [
#     Document(page_content="This is page one with some text about AI and RAG pipelines.", metadata={"page": 1}),
#     Document(page_content="Second page with more text, embeddings and transformers.", metadata={"page": 2}),
# ]

chunks = split_documents(document, chunk_size=50, chunk_overlap=10)
print(f"Total chunks created: {len(chunks)}")

Split 246 documents into 9025 chunks

 Example chunk
Content : LINEAR ALGEBRA...
Metadata : {'producer': 'pikepdf 4.3.1', 'creator': 'ocrmypdf 13.2.0 / Tesseract OCR-PDF 4.1.1', 'creationdate': '2021-08-05T14:17:12+00:00', 'author': '', 'moddate': '2022-01-08T20:19:31+00:00', 'source': '..\\Data\\pdf\\Ma_Siu_Lun,_Ng_Kah_Loon,_Victor_Tan_Linear_Algebra_Concepts_and.pdf', 'total_pages': 246, 'page': 0, 'page_label': 'i'}
Total chunks created: 9025


### Embedding and vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embedding.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self):
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    
#initialise

embedding_manager = EmbeddingManager()
embedding_manager
#print(embedding_manager.model_name)

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


### Vector Store

In [10]:
class VectorStore:
    def __init__(self, collection_name:str ="pdf_documents", persist_directory:str = "../Data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store"""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store")

        #prepare data for chromadb
        id = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):   #create tuple of doc and embeddings
            #generate id
            docs_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            id.append(docs_id)

            #metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #document context
            documents_text.append(doc.page_content)

            #embedding
            embeddings_list.append(embedding.tolist())
        
        #add to collection
        try:
            self.collection.add(
                ids = id,
                embeddings = embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

    
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 4354


In [11]:
chunks

[Document(metadata={'producer': 'pikepdf 4.3.1', 'creator': 'ocrmypdf 13.2.0 / Tesseract OCR-PDF 4.1.1', 'creationdate': '2021-08-05T14:17:12+00:00', 'author': '', 'moddate': '2022-01-08T20:19:31+00:00', 'source': '..\\Data\\pdf\\Ma_Siu_Lun,_Ng_Kah_Loon,_Victor_Tan_Linear_Algebra_Concepts_and.pdf', 'total_pages': 246, 'page': 0, 'page_label': 'i'}, page_content='LINEAR ALGEBRA'),
 Document(metadata={'producer': 'pikepdf 4.3.1', 'creator': 'ocrmypdf 13.2.0 / Tesseract OCR-PDF 4.1.1', 'creationdate': '2021-08-05T14:17:12+00:00', 'author': '', 'moddate': '2022-01-08T20:19:31+00:00', 'source': '..\\Data\\pdf\\Ma_Siu_Lun,_Ng_Kah_Loon,_Victor_Tan_Linear_Algebra_Concepts_and.pdf', 'total_pages': 246, 'page': 0, 'page_label': 'i'}, page_content='Concepts and Techniques on Euclidean Spaces'),
 Document(metadata={'producer': 'pikepdf 4.3.1', 'creator': 'ocrmypdf 13.2.0 / Tesseract OCR-PDF 4.1.1', 'creationdate': '2021-08-05T14:17:12+00:00', 'author': '', 'moddate': '2022-01-08T20:19:31+00:00', '

In [ ]:
def batch_iter(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

batch_limit = 5000

In [15]:
texts = [doc.page_content for doc in chunks]

#generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

for i, (doc_batch, emb_batch) in enumerate(zip(
        batch_iter(chunks, batch_limit),
        batch_iter(embeddings, batch_limit))):

    print(f"Adding batch {i+1}... size {len(doc_batch)}")

    try:    
        #store in vectordb
        vectorstore.add_documents(doc_batch, emb_batch)
    except Exception as e:
        print(f"Error in batch {i+1}: {e}")
        raise



Generating embeddings for 9025 texts...


Batches: 100%|██████████| 283/283 [00:11<00:00, 24.63it/s]


generated embeddings with shape: (9025, 384)
Adding batch 1... size 5000
Adding 5000 documents to vector store
Successfully added 5000 documents to vector store
Total documents in collection: 9354
Adding batch 2... size 4025
Adding 4025 documents to vector store
Successfully added 4025 documents to vector store
Total documents in collection: 13379


### Retrieval Pipeline from VectorStore

In [16]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager) -> None:
        """Initialize the retriever
        Args:
            vector_store: Vectore store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k: int=5, score_threshold: float=0.0) -> List[Dict[str, Any]]:
        """
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshod: Minimum similarity score threshold
            
        Output:
            List of dic containing retrieved documents and metadata"""
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        #generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]   #the first row
        #print(query_embedding)
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #convert distance to similarity score using cosine distance
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


In [17]:
rag_retriever

In [18]:
rag_retriever.retrieve("orthogonal matrices")

Retrieving documents for query: 'orthogonal matrices'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 160.65it/s]

generated embeddings with shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_05f6fcf7_1446',
  'content': 'of orthogonal matrices.',
  'metadata': {'creator': 'ocrmypdf 13.2.0 / Tesseract OCR-PDF 4.1.1',
   'moddate': '2022-01-08T20:19:31+00:00',
   'source': '..\\Data\\pdf\\Ma_Siu_Lun,_Ng_Kah_Loon,_Victor_Tan_Linear_Algebra_Concepts_and.pdf',
   'total_pages': 246,
   'content_length': 23,
   'page_label': 'clxxii',
   'producer': 'pikepdf 4.3.1',
   'creationdate': '2021-08-05T14:17:12+00:00',
   'doc_index': 1446,
   'author': '',
   'page': 171},
  'similarity_score': 0.9116196036338806,
  'distance': 0.08838039636611938,
  'rank': 1},
 {'id': 'doc_80b1fdd4_1387',
  'content': 'Section 5.4 Orthogonal Matrices',
  'metadata': {'moddate': '2022-01-08T20:19:31+00:00',
   'page_label': 'clxx',
   'page': 169,
   'producer': 'pikepdf 4.3.1',
   'creationdate': '2021-08-05T14:17:12+00:00',
   'author': '',
   'source': '..\\Data\\pdf\\Ma_Siu_Lun,_Ng_Kah_Loon,_Victor_Tan_Linear_Algebra_Concepts_and.pdf',
   'total_pages': 246,
   'doc_index': 1387,
  

In [21]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
model = ChatOpenAI(verbose=True, model="gpt-3.5-turbo", temperature=0)

def rag(query, retriever, model, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found."
    
    prompt= f"""Use following context to answer the question with details and examples if possible.
            Context:
            {context}

            Questoin: {query}

            Answer: """
    response = model.invoke([prompt.format(context=context, query=query)])
    return response.content


In [22]:
answer = rag("What is orthogonal matrices", rag_retriever, model)
print(answer)

Retrieving documents for query: 'What is orthogonal matrices'
Top k: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 158.00it/s]

generated embeddings with shape: (1, 384)
Retrieved 3 documents


Orthogonal matrices are square matrices where the rows and columns are orthonormal vectors. This means that the dot product of any two different rows or columns is zero, and the dot product of a row or column with itself is equal to 1. In other words, the columns of an orthogonal matrix are perpendicular to each other and have a length of 1.

For example, consider the following 2x2 orthogonal matrix:

A = [1/sqrt(2)  -1/sqrt(2)]
        [1/sqrt(2)   1/sqrt(2)]

To check if this matrix is orthogonal, we can calculate the dot product of the two columns:

[1/sqrt(2)  1/sqrt(2)] • [ -1/sqrt(2)  1/sqrt(2)] = 0

And the dot product of each column with itself:

[1/sqrt(2)  1/sqrt(2)] • [1/sqrt(2)  1/sqrt(2)] = 1

Therefore, this matrix is orthogonal. Orthogonal matrices have many important properties and applications in mathematics and physics, including in the areas of linear transformations, rotations, and solving systems of equations.


### Enhancing the RAG Pipeline

In [25]:
def rag_advanced(query, retriever, model, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])
    
    #Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = model.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context

    return output

result = rag_advanced("What is win probability in League of Lengends?", rag_retriever, model, top_k=5, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is win probability in League of Lengends?'
Top k: 5, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 145.84it/s]

generated embeddings with shape: (1, 384)
Retrieved 5 documents


Answer: Win probability in League of Legends is a measure of the likelihood of a team winning a match based on various in-game factors and statistics.
Sources: [{'source': '..\\Data\\pdf\\Thesis_xPetu.pdf', 'page': 15, 'score': 0.47665834426879883, 'preview': 'in win probability, similar to win values...'}, {'source': '..\\Data\\pdf\\Thesis_xPetu.pdf', 'page': 6, 'score': 0.4574812650680542, 'preview': 'in win probability associated with a certain...'}, {'source': '..\\Data\\pdf\\Thesis_xPetu.pdf', 'page': 15, 'score': 0.456162691116333, 'preview': 'in sports. In the next section, win probability...'}, {'source': '..\\Data\\pdf\\Thesis_xPetu.pdf', 'page': 15, 'score': 0.4532684087753296, 'preview': 'probability and true proportion of wins....'}, {'source': '..\\Data\\pdf\\Thesis_xPetu.pdf', 'page': 2, 'score': 0.4079629182815552, 'preview': 'win probability added...'}]
Confidence: 0.47665834426879883
Context Preview: in win probability, similar to win values

in win probability associa